# MATRIX — Inter-PLC Attack Analysis (per-event: baseline vs. attack)

For ONE attack experiment, each attack **event** gets its own column of panels.
Within every event, the **baseline** (normal hand-over, grey dashed) is aligned to
that event's onset and overlaid on the **attacked** behaviour (blue), across:

* **WIRE** — the attacked handshake register the victim PLC reads (from the pcap).
* **PHYSICAL** — the hand-over actuators/sensors, in process order (from
  `results/matrix.db`). Signals that never change in the run (a logging bug
  upstream) are auto-dropped so no empty lanes are shown.

Because each event is aligned to its own onset, baseline and attack are directly
comparable at every event (no cross-run drift). Set `EXPERIMENT_ID` and Run All.


In [17]:
import os
from pathlib import Path
import numpy as np, pandas as pd, json, sqlite3, shutil, tempfile
import matplotlib as mpl, matplotlib.pyplot as plt
from datetime import datetime
from scapy.all import rdpcap, IP, TCP, Raw, Ether

In [18]:
def _base():
    """Repo root: nearest ancestor containing a 'results' folder (works from
    notebooks/ or the repo root)."""
    d = Path(os.getcwd())
    for cand in [d.parent, d, *d.parents]:
        if (cand / "results").is_dir():
            return cand
    return d.parent

In [19]:
# GLOBAL CONSTANTS
EXPERIMENT_ID = "t3_suppress_0001"
BASELINE_ID   = "baseline6wps_0001"
PRE, POST     = 6.0, 8.0          # seconds shown before onset / after the window

BASE    = _base()
RESULTS = os.path.join(BASE, "results")
DB      = os.path.join(RESULTS, "matrix.db")
def expdir(r): return os.path.join(RESULTS, r)
FIGDIR  = os.path.join(expdir(EXPERIMENT_ID), "figures"); os.makedirs(FIGDIR, exist_ok=True)

AB_IP = "192.168.50.20"
CLIENT_IP = {0:"192.168.50.10", 1:"192.168.50.50", 2:"192.168.50.50"}

# Ideal hand-over signals per attacked register (tag, label). Dead ones (never
# change in the run) are dropped automatically, so leave the transferUnitAt*
# switches here — they will appear once the middleware polling bug is fixed.
HANDOVER = {
    0: [("motorTurntable","turntable motor"),
        ("inPosition","turntable in-position"),
        ("transferUnitAtTurntable","transfer @ turntable")],
    1: [("transferUnitAtConveyor","transfer @ conveyor"),
        ("workpieceOnConveyor","workpiece on conveyor"),
        ("conveyorBelt","conveyor belt motor")],
    2: [("transferUnitAtConveyor","transfer @ conveyor"),
        ("workpieceOnConveyor","workpiece on conveyor"),
        ("conveyorBelt","conveyor belt motor")],
}
REG_LABEL = {0:"reg0", 1:"reg1", 2:"reg2"}

mpl.rcParams.update({"figure.dpi":120,"savefig.dpi":300,"font.family":"serif",
                     "font.size":9,"axes.titlesize":10,"savefig.bbox":"tight"})
BLUE="#0072B2"; GREEN="#009E73"; RED="#C1121F"; GREY="#8A8A8A"
def u16(b,i): return (b[i]<<8)|b[i+1]
print("repo root:", BASE, "| DB:", DB, "(exists:", os.path.exists(DB), ")")

repo root: /home/chabeli/Private/Matrix-ICS-Testbed | DB: /home/chabeli/Private/Matrix-ICS-Testbed/results/matrix.db (exists: False )


In [20]:
# --- Attacker plane ------------------------------------------------------
alog=json.load(open(os.path.join(expdir(EXPERIMENT_ID),"attack_log.json")))
cfg=alog["config"]
REG=int(cfg["register"]); ACTION=cfg["action"]; TRANS=cfg["transition"]; PAIR=cfg["pair"]
SIGVAL=int(cfg["signal_value"]); client_ip=CLIENT_IP[REG]
signals_all=HANDOVER[REG]
events=alog["attack_events"]
def ep(iso): return datetime.fromisoformat(iso).timestamp()
onsets=[(ep(e["onset_wall"]),ep(e["close_wall"])) for e in events]
HOLD=float(cfg.get("hold_s",15.0))
print(f"{EXPERIMENT_ID}: {PAIR} {TRANS} {ACTION}  register={REG}  signal_value={SIGVAL}  events={len(events)}")

t3_suppress_0001: tm221-ab ready suppress  register=1  signal_value=1  events=3


In [21]:
# --- Loaders (wire + physical, absolute epoch) --------------------------
def _macs(pk, cip):
    cmac=amac=None
    for p in pk:
        if IP in p and TCP in p and Ether in p:
            if p[IP].src==cip and p[IP].dst==AB_IP and cmac is None: cmac=p[Ether].src
            if p[IP].src==AB_IP and amac is None: amac=p[Ether].src
            if cmac and amac: break
    return cmac, amac

def _reg_timeline(pk, reg, ether_dst=None, ether_src=None):
    pend={}; out=[]
    for p in pk:
        if IP not in p or TCP not in p or Raw not in p: continue
        tcp=p[TCP]; load=bytes(p[Raw].load)
        if len(load)<8: continue
        txid=u16(load,0); fc=load[7]
        to_ab=(p[IP].dst==AB_IP and tcp.dport==502); from_ab=(p[IP].src==AB_IP and tcp.sport==502)
        if to_ab and fc==0x03 and len(load)>=12: pend[(tcp.sport,txid)]=(u16(load,8),u16(load,10))
        elif from_ab and fc==0x03 and len(load)>=9:
            if ether_dst and p[Ether].dst!=ether_dst: continue
            if ether_src and p[Ether].src!=ether_src: continue
            sq=pend.get((tcp.dport,txid))
            if sq and sq[0]<=reg<sq[0]+sq[1]:
                voff=9+(reg-sq[0])*2
                if voff+2<=len(load): out.append((float(p.time),u16(load,voff)))
    return out

def _db():
    try: return sqlite3.connect(f"file:{os.path.abspath(DB)}?mode=ro",uri=True)
    except sqlite3.OperationalError:
        tmp=os.path.join(tempfile.gettempdir(),"m_ro.db"); shutil.copy(DB,tmp)
        return sqlite3.connect(f"file:{tmp}?mode=ro",uri=True)

def load_run(run_id):
    d=expdir(run_id)
    pf=[f for f in os.listdir(d) if f.endswith((".pcap",".pcapng"))][0]
    pk=rdpcap(os.path.join(d,pf)); cmac,amac=_macs(pk,client_ip)
    wire_c=_reg_timeline(pk,REG,ether_dst=cmac)
    wire_a=_reg_timeline(pk,REG,ether_src=amac)
    tags=[t for t,_ in signals_all]
    con=_db()
    ph=pd.read_sql_query("SELECT ts_utc,tag,value,vtype FROM state_events WHERE run_id=? AND tag IN (%s) ORDER BY seq"
                         % ",".join("?"*len(tags)), con, params=[run_id]+tags); con.close()
    ph["e"]=ph["ts_utc"].map(lambda s: datetime.fromisoformat(s).timestamp())
    ph["num"]=np.where(ph.vtype=="bool", ph.value.map(lambda v:1.0 if str(v)=="True" else 0.0),
                       pd.to_numeric(ph.value,errors="coerce"))
    return {"wire_c":wire_c,"wire_a":wire_a,"ph":ph}

A=load_run(EXPERIMENT_ID); B=load_run(BASELINE_ID)

# Drop dead signals (<=1 distinct value in the attack run) -> no empty lanes.
def alive(run, tag):
    v=run["ph"][run["ph"].tag==tag]["num"]
    return v.nunique()>1
signals=[(t,l) for (t,l) in signals_all if alive(A,t) or alive(B,t)]
dropped=[t for (t,l) in signals_all if (t,l) not in signals]
print("live hand-over signals:", [t for t,_ in signals])
if dropped: print("DROPPED (dead/unlogged — fix middleware polling):", dropped)

FileNotFoundError: [Errno 2] No such file or directory: '/home/chabeli/Private/Matrix-ICS-Testbed/results/matrix.db'

In [ ]:
# --- helpers: windowed step series relative to a center time ------------
def win_series(series, center, lo, hi):
    """series=[(epoch,val)] -> (xs,ys) within [center+lo, center+hi], x relative
    to center, step-held, seeded with the last value before the window."""
    xs=[]; ys=[]; prev=None
    for e,v in series:
        if e < center+lo: prev=(e,v); continue
        if e > center+hi: break
        if not xs:
            xs.append(lo); ys.append(prev[1] if prev else v)
        xs.append(e-center); ys.append(v)
    if not xs:
        pv=prev[1] if prev else (series[-1][1] if series else 0)
        return np.array([lo,hi]), np.array([pv,pv])
    xs.append(hi); ys.append(ys[-1])
    return np.array(xs), np.array(ys)

def phys_series(run, tag):
    g=run["ph"][run["ph"].tag==tag].sort_values("e")
    return list(zip(g.e.to_numpy(), g.num.to_numpy()))

# baseline alignment: find the representative normal hand-over = a reg rising edge
# to SIGVAL in the baseline wire; align baseline to each attack event by that edge.
def rising_edges(series, val):
    es=[]; prev=None
    for e,v in series:
        if v==val and prev!=val: es.append(e)
        prev=v
    return es
b_edges=rising_edges(B["wire_c"], SIGVAL)
B_CENTER = b_edges[len(b_edges)//2] if b_edges else (B["ph"].e.median() if len(B["ph"]) else 0)
print(f"baseline normal hand-overs found: {len(b_edges)}; using representative at index {len(b_edges)//2}")

## Per-event panels — baseline (grey) vs. attack (blue)

In [ ]:
LO, HI = -PRE, HOLD+POST
nrows=1+len(signals); ncols=len(onsets)
fig,axs=plt.subplots(nrows,ncols,figsize=(2.3*ncols+1.0, 0.85*nrows+1.0),
                     sharex=True, sharey="row", squeeze=False,
                     gridspec_kw={"hspace":0.25,"wspace":0.12})
fig.suptitle(f"{EXPERIMENT_ID} — {PAIR} {TRANS} {ACTION}: baseline (grey) vs attack (blue), per event",
             fontsize=9.5)

for j,(o,c) in enumerate(onsets,0):
    win=c-o
    # ---- WIRE row ----
    ax=axs[0][j]
    ax.axvspan(0, win, color=RED, alpha=0.10)
    xb,yb=win_series(B["wire_c"], B_CENTER, LO, HI)
    ax.step(xb,yb,where="post",color=GREY,lw=1.1,ls="--",alpha=0.85)
    xa,ya=win_series(A["wire_c"], onsets[j][0], LO, HI)
    ax.step(xa,ya,where="post",color=BLUE,lw=1.5)
    if ACTION in ("forge","replay"):
        xt,yt=win_series(A["wire_a"], onsets[j][0], LO, HI)
        ax.step(xt,yt,where="post",color=GREEN,lw=1.0,ls=":")
    ax.set_ylim(-0.3,2.3); ax.set_yticks([0,1,2]); ax.set_yticklabels(["0","1","2"],fontsize=6)
    ax.set_title(f"event #{j+1}", fontsize=8)
    if j==0: ax.set_ylabel(f"WIRE {REG_LABEL[REG]}", fontsize=7)
    # ---- PHYSICAL rows ----
    for k,(tag,label) in enumerate(signals):
        ax=axs[1+k][j]; ax.axvspan(0,win,color=RED,alpha=0.10)
        xb,yb=win_series(phys_series(B,tag), B_CENTER, LO, HI)
        ax.step(xb,0.1+0.8*yb,where="post",color=GREY,lw=1.0,ls="--",alpha=0.85)
        xa,ya=win_series(phys_series(A,tag), onsets[j][0], LO, HI)
        ax.step(xa,0.1+0.8*ya,where="post",color=BLUE,lw=1.4)
        ax.set_ylim(-0.05,1.05); ax.set_yticks([])
        if j==0: ax.set_ylabel(label, fontsize=7)
    axs[-1][j].set_xlabel("t\u2212onset (s)", fontsize=7)

# one shared legend
from matplotlib.lines import Line2D
leg=[Line2D([0],[0],color=GREY,ls="--",label="baseline"),
     Line2D([0],[0],color=BLUE,label="under attack"),
     mpl.patches.Patch(color=RED,alpha=0.15,label="attack window")]
if ACTION in ("forge","replay"): leg.insert(2,Line2D([0],[0],color=GREEN,ls=":",label="AB true"))
fig.legend(handles=leg,loc="lower center",ncol=len(leg),fontsize=7,bbox_to_anchor=(0.5,-0.04))
savename=f"attack_perevent_{EXPERIMENT_ID}"
for ext in ("pdf","png"): fig.savefig(os.path.join(FIGDIR,f"{savename}.{ext}"))
print("saved:", savename); plt.show()

In [ ]:
# --- Impact table (per event) -------------------------------------------
rows=[]
for i,(o,c) in enumerate(onsets,1):
    wc=[v for t,v in A["wire_c"] if o<=t<=c]
    ph=A["ph"][(A["ph"].e>=o)&(A["ph"].e<=c)]
    rows.append({"event":i,"window_s":round(c-o,1),
                 "victim_saw":dict(pd.Series(wc).value_counts()) if wc else {},
                 "reached_signal(%d)"%SIGVAL: (SIGVAL in wc),
                 "handover_transitions_in_window":len(ph)})
imp=pd.DataFrame(rows); display(imp)
imp.to_csv(os.path.join(FIGDIR,"impact_summary.csv"),index=False); print("saved impact_summary.csv")